In [ ]:
import sys
import os
import logging

# Adiciona o diretório raiz do projeto ao sys.path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

import requests
from services.database import get_collection

# Substituir pelo seu token correto
TOKEN_RD = '65171203c468ec001837fdca'

def listar_ganhos():
    headers = {"accept": "application/json"}
    offset = True
    page = 1
    limit = 200

    # Lista para armazenar os dados das negociações
    dados_negociacoes = []

    # Loop para obter todas as páginas da API
    while offset:
        url = f'https://crm.rdstation.com/api/v1/deals?token={TOKEN_RD}&page={page}&limit={limit}&win=true'
        print(f'Requisitando URL: {url}')
        response = requests.get(url, headers=headers)

        # Verifica se a resposta foi bem-sucedida
        if response.status_code == 200:
            data = response.json()
            dados_negociacoes.extend(data['deals'])
            print(f"Página {page} - Mais negociações: {data['has_more']} - Total: {len(dados_negociacoes)}")

            # Atualiza o valor de offset e incrementa a página
            offset = data['has_more']
            page += 1
        else:
            print(f"Erro ao obter dados: {response.status_code}")
            break

    # Salvar os dados no arquivo JSON
    return dados_negociacoes

def iniciar_warmup(id, name, amount_total, organization):
    novo_warmup = {
        'negocio_id': id,
        'name': name,
        'organization': organization,
        'amount_total': amount_total,
        'status': 'Finalizado'
    }
    print(f'Iniciando warmup para: {novo_warmup}')
    collection = get_collection("warmup_projetos")
    collection.insert_one(novo_warmup)

ganhos = listar_ganhos()
print(f'Ganhos obtidos: {ganhos}')
for ganho in ganhos:
    id = ganho['_id']
    name = ganho['name']
    amount_total = ganho['amount_total']
    organization = ganho['organization']['name']
    iniciar_warmup(id, name, amount_total, organization)